# Latent Representation Learning

This notebook implements the latent learning fusion method introduced in https://pubmed.ncbi.nlm.nih.gov/31021792/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from importlib import reload
import seaborn as sns
import os, sys
import torch
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
target_encoder = LabelEncoder()
onehot_encoder = OneHotEncoder(sparse=False)

def load_data(impute_method = 'Mean-Mode', target = 'NACCAD3'):
    uds = pd.read_csv("../data/data_imputed/{}/uds.csv".format(impute_method))
    uds['datetime'] = pd.to_datetime(uds['datetime'])
    uds = uds.dropna(subset=[target, 'EDUC'])
    print("UDS Shape:  ", uds.shape)
    print("Target Distribution: {}\n".format(Counter(uds[target])))
    uds[target] = target_encoder.fit_transform(uds[target])
    onehot_encoder.fit(uds[target].values.reshape(-1, 1))
    mri = pd.read_csv("../data/data_imputed/{}/mri.csv".format(impute_method))
    mri['datetime'] = pd.to_datetime(mri['datetime'])
    
    csf = pd.read_csv("../data/data_imputed/{}/csf.csv".format(impute_method))
    return uds, mri, csf

uds_dict = pd.read_csv("../data/data_dictionary/uds_feature_dictionary_cleaned.csv")
mri_dict = pd.read_csv("../data/data_dictionary/mri_feature_dictionary_cleaned.csv") 

uds_drop_columns = ['NACCID', 'NACCADC', 'NACCVNUM', 'datetime', 'SEX', 'NACCAGE','EDUC', 
                    'NACCUDSD', 'NACCALZP', 'NACCAD3', 'NACCAD5']
mri_drop_columns = ['NACCID', 'NACCVNUM', 'datetime', 'datetime_UDS', 'timediff', 'within-a-year']
csf_drop_columns = ['NACCID', 'CSFABMD', 'CSFTTMD', 'CSFPTMD']

In [ ]:
uds, mri, csf = load_data(impute_method = 'Mean-Mode', target = 'NACCAD3')
print(uds.shape, mri.shape, csf.shape)

# Latent Representation Learning 

In [ ]:
common_pid = set(uds['NACCID']).intersection(mri['NACCID']).intersection(csf['NACCID'])
cpidTr, cpidTe = train_test_split(np.array(list(common_pid)), test_size = 0.3)

def train_test_split_modality(df, cpidTr, cpidTe):
    dfV = df[~df['NACCID'].isin(common_pid)]
    idx = dfV.index.to_frame()
    iTr, iTe = train_test_split(idx[0], test_size = 0.3)
    dfVTr, dfVTe = dfV.loc[iTr], dfV.loc[iTe]
    dfTr = pd.concat([df[df['NACCID'].isin(cpidTr)], dfVTr], axis=0).reset_index(drop=True)
    dfTe = pd.concat([df[df['NACCID'].isin(cpidTe)], dfVTe], axis=0).reset_index(drop=True)
    return dfTr, dfTe

mri, csf = mri[mri['NACCID'].isin(uds['NACCID'])], csf[csf['NACCID'].isin(uds['NACCID'])]
uds, mri, csf = uds.sort_values('NACCID'), mri.sort_values('NACCID'), csf.sort_values('NACCID')

udsTr, udsTe = train_test_split_modality(uds, cpidTr, cpidTe)
mriTr, mriTe = train_test_split_modality(mri, cpidTr, cpidTe)
csfTr, csfTe = train_test_split_modality(csf, cpidTr, cpidTe) 

In [ ]:
def prepare_X_Y(udsT, mriT, csfT, scalerU=None, scalerM=None, scalerC=None):
    XU, XM, XC = udsT.drop(uds_drop_columns, axis=1), mriT.drop(mri_drop_columns, axis=1), csfT.drop(csf_drop_columns, axis=1)
    if scalerU is None:
        scalerU, scalerM, scalerC = StandardScaler(), StandardScaler(), StandardScaler()
        XU, XM, XC = scalerU.fit_transform(XU).T, scalerM.fit_transform(XM).T, scalerC.fit_transform(XC).T
    else:
        XU, XM, XC = scalerU.transform(XU).T, scalerM.transform(XM).T, scalerC.transform(XC).T
    XU, XM, XC = torch.tensor(XU).float(), torch.tensor(XM).float(), torch.tensor(XC).float()

    YU, YM, YC = udsT['NACCAD3'], uds.merge(mriT, on='NACCID')['NACCAD3'], uds.merge(csfT, on='NACCID')['NACCAD3']
    YU, YM, YC = onehot_encoder.transform(YU.values.reshape(-1, 1)),onehot_encoder.transform(YM.values.reshape(-1, 1)),onehot_encoder.transform(YC.values.reshape(-1, 1))
    YU, YM, YC = torch.tensor(YU.T).float(), torch.tensor(YM.T).float(), torch.tensor(YC.T).float()
    return XU, XM, XC, YU, YM, YC, scalerU, scalerM, scalerC

In [ ]:
def train_weights(W, X, H, E, Q, mu, verbose = False):
    for i in range(100):
        G = torch.diag((W**2).sum(1))
        W_new = torch.inverse(X.matmul(X.T) + (2*beta) / mu * G).matmul(X).matmul((H + E - Q / mu).T)
        if ((W_new - W)**2).sum() < 1e-6:
            break
        W = W_new
    if verbose and i == 100:
        print("Not converged")
    return W

def update_E(E, W, X, H, Q, mu):
    E_new = torch.zeros_like(E)
    temp = W.T.matmul(X) - H + Q / mu
    E_new = E_new + (temp > gamma / mu).int() * temp - gamma/mu
    E_new = E_new + (temp < -gamma / mu).int() * temp + gamma/mu
    return E_new

def calculate_accuracy(XU, XM, XC, YU, YM, YC, WU, WM, WC, P, nH):
    HU, HM, HC = WU.T.matmul(XU), WM.T.matmul(XM), WC.T.matmul(XC)
    H = (HU[:,:nH] + HM[:,:nH] + HC[:,:nH]) / 3
    YU_hat = P.matmul(torch.concat([H, HU[:,nH:]], axis=1)).argmax(axis=0)
    YM_hat = P.matmul(torch.concat([H, HM[:,nH:]], axis=1)).argmax(axis=0)
    YC_hat = P.matmul(torch.concat([H, HC[:,nH:]], axis=1)).argmax(axis=0)
    return "U-acc: {:.2f}%\tM-acc: {:.2f}%\tC-acc: {:.2f}%".format(
        (YU_hat == YU.argmax(0)).float().mean() * 100,
        (YM_hat == YM.argmax(0)).float().mean() * 100,
        (YC_hat == YC.argmax(0)).float().mean() * 100)
    
# maxiter, stop, rho, max_mu =1000, 1, 1.5, 1e6
def train_latent_features(maxiter=1000, stop = 1, rho = 1.5, mu=1e-2, max_mu = 1e6):
#     Train individual weights
    for i in range(maxiter):
        H = torch.randn(kH, nH)
        P = torch.randn(3, kH)
        WU, WM, WC = torch.randn(kU, kH), torch.randn(kM, kH), torch.randn(kC, kH)
        HU, EU, QU = torch.randn(kH, XU.shape[1]-nH), torch.randn(kH, XU.shape[1]), torch.ones(kH, XU.shape[1])
        HM, EM, QM = torch.randn(kH, XM.shape[1]-nH), torch.randn(kH, XM.shape[1]), torch.ones(kH, XM.shape[1])
        HC, EC, QC = torch.randn(kH, XC.shape[1]-nH), torch.randn(kH, XC.shape[1]), torch.ones(kH, XC.shape[1])

        WU = train_weights(WU, XU, torch.concat([H, HU], axis=1), EU, QU, mu)
        WM = train_weights(WM, XM, torch.concat([H, HM], axis=1), EM, QM, mu)
        WC = train_weights(WC, XC, torch.concat([H, HC], axis=1), EC, QC, mu)

        EU_new = update_E(EU, WU, XU, torch.concat([H, HU], axis=1), QU, mu)
        EM_new = update_E(EM, WM, XM, torch.concat([H, HM], axis=1), QM, mu)
        EC_new = update_E(EC, WC, XC, torch.concat([H, HC], axis=1), QC, mu)

        HU = torch.inverse((P.T.matmul(P)) + mu * torch.eye(kH)).matmul(
            P.T.matmul(YU[:,nH:]) + mu * (WU.T.matmul(XU[:,nH:]) - EU[:,nH:] + QU[:,nH:] / mu))
        HM = torch.inverse((P.T.matmul(P)) + mu * torch.eye(kH)).matmul(
            P.T.matmul(YM[:,nH:]) + mu * (WM.T.matmul(XM[:,nH:]) - EM[:,nH:] + QM[:,nH:] / mu))
        HC = torch.inverse((P.T.matmul(P)) + mu * torch.eye(kH)).matmul(
            P.T.matmul(YC[:,nH:]) + mu * (WC.T.matmul(XC[:,nH:]) - EC[:,nH:] + QC[:,nH:] / mu))

        # train global parameters
        H = torch.inverse(P.T.matmul(P) + mu * torch.eye(kH) * 3).matmul(P.T.matmul(YU[:,:nH]) + mu * (
        (WU.T.matmul(XU[:,:nH]) - EU[:,:nH] + QU[:,:nH] / mu) + 
        (WM.T.matmul(XM[:,:nH]) - EM[:,:nH] + QM[:,:nH] / mu) + 
        (WC.T.matmul(XC[:,:nH]) - EC[:,:nH] + QC[:,:nH] / mu)
        ))

        H_all = torch.concat([H, HU, HM, HC], axis=1)
        Y_all = torch.concat([YU[:,:nH], YU[:,nH:], YM[:,nH:], YC[:,nH:]],axis=1)
        P = Y_all.matmul(H_all.T).matmul(torch.inverse(H_all.matmul(H_all.T) + eta * torch.eye(kH)))

        QU = QU + mu * (WU.T.matmul(XU) - torch.concat([H, HU], axis=1) - EU)
        QM = QM + mu * (WM.T.matmul(XM) - torch.concat([H, HM], axis=1) - EM)
        QC = QC + mu * (WC.T.matmul(XC) - torch.concat([H, HC], axis=1) - EC)

        mu = min(rho * mu, max_mu)

        errU = torch.abs(WU.T.matmul(XU) - torch.concat([H, HU], axis=1) - EU).max()
        errM = torch.abs(WM.T.matmul(XM) - torch.concat([H, HM], axis=1) - EM).max()
        errC = torch.abs(WC.T.matmul(XC) - torch.concat([H, HC], axis=1) - EC).max()
        error = max(errU, errM, errC)
        if error < stop:
            break
        if i % 20 == 0:
            print("Iter {}:  error-{:.3f}\t{}".format(i, error, 
                                                    calculate_accuracy(XU, XM, XC, YU, YM, YC, WU, WM, WC, P, len(cpidTr))))
    if i == maxiter:
        print("Not Converged")
    return WU, WM, WC, P

In [ ]:
XU, XM, XC, YU, YM, YC, scalerU, scalerM, scalerC = prepare_X_Y(udsTr, mriTr, csfTr)
kU, kM, kC = XU.shape[0], XM.shape[0], XC.shape[0]

nH, kH = len(cpidTr), 20
lamb, beta, gamma, eta = 0.1, 0.1, 0.1, 0.1

WU, WM, WC, P = train_latent_features(maxiter=5000, stop = 1, rho = 1.5, mu=1e-2, max_mu = 1e6)

In [ ]:
XUTe, XMTe, XCTe, YUTe, YMTe, YCTe, scalerU, scalerM, scalerC = prepare_X_Y(udsTe, mriTe, csfTe, scalerU, scalerM, scalerC)

In [ ]:
print(calculate_accuracy(XUTe, XMTe, XCTe, YUTe, YMTe, YCTe, WU, WM, WC, P, len(cpidTr)))